In [1]:
# NB_00_Setup — Run once before anything else
# Creates two control tables:
#
# 1. pipeline_log     — tracks every pipeline run (success or failure)
# 2. quarantine_log   — tracks every file that was quarantined
#
# WHY TWO SEPARATE TABLES:
# pipeline_log answers: "did the pipeline run successfully?"
# quarantine_log answers: "which specific files failed and why?"
# These are different questions and should be in different tables.
# Data Activator monitors quarantine_log to fire alerts.

print("Creating CS3 control tables...")

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 3, Finished, Available, Finished, False)

Creating CS3 control tables...


In [2]:
# pipeline_log records one row per pipeline run

spark.sql("""
    CREATE TABLE IF NOT EXISTS pipeline_log (
        run_id            STRING    NOT NULL,
        pipeline_name     STRING    NOT NULL,
        run_timestamp     TIMESTAMP NOT NULL,
        status            STRING    NOT NULL,
        files_found       INT,
        files_processed   INT,
        files_quarantined INT,
        rows_loaded       LONG,
        duration_seconds  INT,
        error_message     STRING,
        notes             STRING
    )
    USING DELTA
""")

print("pipeline_log created ✅")
spark.sql("DESCRIBE pipeline_log").show(truncate=False)

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 4, Finished, Available, Finished, False)

pipeline_log created ✅
+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|run_id           |string   |NULL   |
|pipeline_name    |string   |NULL   |
|run_timestamp    |timestamp|NULL   |
|status           |string   |NULL   |
|files_found      |int      |NULL   |
|files_processed  |int      |NULL   |
|files_quarantined|int      |NULL   |
|rows_loaded      |bigint   |NULL   |
|duration_seconds |int      |NULL   |
|error_message    |string   |NULL   |
|notes            |string   |NULL   |
+-----------------+---------+-------+



In [3]:
%%sql
DROP TABLE IF EXISTS quarantine_log;


StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
# quarantine_log records one row per quarantined file
# Data Activator monitors this table
# If quarantine_log has any new rows after a run → alert fires

spark.sql("""
    CREATE TABLE IF NOT EXISTS quarantine_log (
        quarantine_id     STRING    NOT NULL,
        run_id            STRING    NOT NULL,
        file_name         STRING    NOT NULL,
        file_path         STRING    NOT NULL,
        file_extension    STRING,
        quarantine_reason STRING    NOT NULL,
        quarantine_ts     TIMESTAMP NOT NULL,
        file_size_bytes   LONG,
        resolved          BOOLEAN  ,
        resolved_by       STRING,
        resolved_ts       TIMESTAMP,
        notes             STRING
    )
    USING DELTA
""")

print("quarantine_log created ✅")
spark.sql("DESCRIBE quarantine_log").show(truncate=False)

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 6, Finished, Available, Finished, False)

quarantine_log created ✅
+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|quarantine_id    |string   |NULL   |
|run_id           |string   |NULL   |
|file_name        |string   |NULL   |
|file_path        |string   |NULL   |
|file_extension   |string   |NULL   |
|quarantine_reason|string   |NULL   |
|quarantine_ts    |timestamp|NULL   |
|file_size_bytes  |bigint   |NULL   |
|resolved         |boolean  |NULL   |
|resolved_by      |string   |NULL   |
|resolved_ts      |timestamp|NULL   |
|notes            |string   |NULL   |
+-----------------+---------+-------+



In [5]:
spark.sql("""
    ALTER TABLE quarantine_log
SET TBLPROPERTIES (
  'delta.feature.allowColumnDefaults' = 'supported'
);

""")

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 7, Finished, Available, Finished, False)

DataFrame[]

In [6]:
spark.sql("""
    ALTER TABLE quarantine_log
    ALTER COLUMN resolved SET DEFAULT FALSE;
""")

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 8, Finished, Available, Finished, False)

DataFrame[]

In [7]:
# Verify all 4 files are visible in the incoming folder
# If any file is missing — stop and upload it before proceeding

from notebookutils import mssparkutils

print("Files in incoming/ folder:")
files = mssparkutils.fs.ls("Files/incoming/")

for f in files:
    print(f"  {f.name:35s}  {f.size:>10,} bytes")

print(f"\nTotal files found: {len(files)}")

# Expected:
#   listings_usa.csv         ~137,000 bytes
#   listings_europe.json     ~307,000 bytes
#   listings_apac.xlsx       ~38,000  bytes
#   corrupted_data.txt       ~70      bytes

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 9, Finished, Available, Finished, False)

Files in incoming/ folder:
  cs3_dataset_reference.json                3,277 bytes
  listings_apac.xlsx                       38,830 bytes
  listings_europe.json                    307,122 bytes
  listings_usa.csv                        136,692 bytes

Total files found: 4


In [8]:
# CS3 uses 2 Lakehouses:
# Bronze — Blackwood_Bronze_Lakehouse (ingestion + control tables)
# Silver — Blackwood_Silver_Lakehouse (unified clean listings)
#
# This cell confirms both are reachable from this notebook

# Bronze check — should see pipeline_log and quarantine_log
print("Bronze Lakehouse tables:")
spark.sql("SHOW TABLES").show(truncate=False)

# Silver check — cross-Lakehouse read
print("\nSilver Lakehouse accessible:")
try:
    spark.read.format("delta").table(
        "Blackwood_Silver_Lakehouse.silver_listings"
    )
    print("  silver_listings exists")
except Exception:
    print("  silver_listings does not exist yet — will be created in NB_04")

StatementMeta(, d22cd5ba-ec3e-4fa2-975c-b32054fdc3f6, 10, Finished, Available, Finished, False)

Bronze Lakehouse tables:
+--------------------------+--------------+-----------+
|namespace                 |tableName     |isTemporary|
+--------------------------+--------------+-----------+
|blackwood_bronze_lakehouse|pipeline_log  |false      |
|blackwood_bronze_lakehouse|quarantine_log|false      |
+--------------------------+--------------+-----------+


Silver Lakehouse accessible:
  silver_listings does not exist yet — will be created in NB_04
